In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------- MLP 3->16->1 --------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

# -------- Init state x_ij --------
def init_state(N):
    X = torch.zeros(N, N)
    for i in range(N):
        X[i, i] = 1
    return X

# -------- Free capacities --------
def compute_free_capacities(X, weights, W):
    load = torch.matmul(weights, X)
    return W - load

# -------- Select item --------
def select_item(policy, X, weights, free_caps, T):
    N = X.shape[0]
    bin_indices = torch.argmax(X, dim=1)
    c_bi = free_caps[bin_indices]
    T_vec = torch.full((N,), T)
    features = torch.stack([weights, c_bi, T_vec], dim=1)
    logits = policy(features).squeeze(-1)
    probs = torch.softmax(logits, dim=0)
    dist = torch.distributions.Categorical(probs)
    i = dist.sample()
    return i

# -------- Select bin --------
def select_bin(policy, i, X, weights, free_caps, T):
    N = X.shape[0]
    w_i = weights[i]
    weight_vec = torch.full((N,), w_i)
    T_vec = torch.full((N,), T)
    features = torch.stack([weight_vec, free_caps, T_vec], dim=1)
    logits = policy(features).squeeze(-1)
    mask = free_caps >= w_i
    logits[~mask] = -1e9
    probs = torch.softmax(logits, dim=0)
    dist = torch.distributions.Categorical(probs)
    j = dist.sample()
    return j

# -------- Apply move --------
def apply_move(X, i, j):
    X_new = X.clone()
    old_bin = torch.argmax(X_new[i])
    X_new[i, old_bin] = 0
    X_new[i, j] = 1
    return X_new

# -------- Energy --------
def energy(X):
    used_bins = torch.sum(X, dim=0) > 0
    return torch.sum(used_bins)

# -------- Metropolis --------
def metropolis_step(X, X_new, T):
    E_old = energy(X)
    E_new = energy(X_new)
    delta = E_new - E_old
    if delta <= 0:
        return X_new
    prob = torch.exp(-delta / T)
    if torch.rand(1) < prob:
        return X_new
    return X

# -------- Neural SA loop --------
def neural_sa(weights, W, steps=100, T0=1.0):
    N = len(weights)
    item_policy = MLP()
    bin_policy = MLP()
    X = init_state(N)
    history = []

    for t in range(steps):
        T = T0 * (1 - t/steps)
        free_caps = compute_free_capacities(X, weights, W)
        i = select_item(item_policy, X, weights, free_caps, T)
        j = select_bin(bin_policy, i, X, weights, free_caps, T)
        X_new = apply_move(X, i, j)
        X = metropolis_step(X, X_new, T)
        history.append(energy(X).item())

    return X, history

In [2]:
# -------- Exemple --------
N = 5
weights = torch.tensor([2.0, 3.0, 1.5, 4.0, 2.5])
W = .0

X_final, history = neural_sa(weights, W, steps=50, T0=1.0)

print("État final x_ij :\n", X_final)
print("Nombre de bins utilisés :", energy(X_final).item())
print("Historique des énergies :", history)

État final x_ij :
 tensor([[0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1.]])
Nombre de bins utilisés : 2
Historique des énergies : [4, 4, 3, 2, 2, 3, 3, 3, 4, 4, 4, 4, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
